In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# nmi_scores_file = "/scicore/home/nimwegen/pachko/BGee/run4daan/run_05.01.2026/nmi.out"
nmi_scores_file = "nmi.out"

In [ ]:
method_dict = {'min_dist_clst_of_cellTypeName': "Unsupervised Bonsai-clustering", 
               'F_1e-2_annot_based_clst_of_cellTypeName': 'Annotation-guided clustering, only big improvements',
               'T_1e-4_annot_based_clst_of_cellTypeName': 'Annotation-guided clustering, prohibit small clusters',
               'F_1e-4_annot_based_clst_of_cellTypeName': 'Supervised Bonsai-clustering',
               'leiden': 'Leiden, tutorial-suggested parameters',
               'leiden_default': 'Leiden',
               'louvain_default': 'Louvain'
              }

In [ ]:
# Read the file
with open(nmi_scores_file) as f:
    lines = f.readlines()

rows = []
current_dataset = None
# Parse the file
for line in lines:
    line = line.rstrip()
    if not line:
        continue
    # Dataset name: no indentation
    if not line.startswith("\t"):
        current_dataset = line
    else:
        parts = line.split()
        method = parts[0]
        nmi = float(parts[1])
        if len(parts) > 2:
            time = float(parts[2])
        else:
            time = np.nan
        rows.append({
                "dataset": current_dataset,
                "method": method,
                "nmi": nmi,
                "time": time
            })
        

# Create DataFrame
df = pd.DataFrame(rows)
df = df[~df["method"].str.contains(r"_annot_cluster_n|^annot_based_clst_of_cellTypeName$", na=False)]
df["method"] = df["method"].map(method_dict)
df["dataset"] = (
    df["dataset"]
    .str.removesuffix("_droplet_based")
    .str.replace("ERP", "", regex=False))
df

In [ ]:
np.unique(df['method'].values)

In [ ]:
# Compute average NMI per dataset and order datasets
avg_nmi = df.groupby("dataset")["nmi"].mean().sort_values()
avg_nmi
ordered_datasets = avg_nmi.index.tolist()

xpos = {ds: i for i, ds in enumerate(ordered_datasets)}
df["x"] = df["dataset"].map(xpos)

In [ ]:
# methods_to_keep = [
#     "F_1e-2_annot_based_clst_of_cellTypeName",
#     "F_1e-4_annot_based_clst_of_cellTypeName",
#     "T_1e-4_annot_based_clst_of_cellTypeName",
#     "leiden",
#     "leiden_default",
#     "louvain_default"
# ]
        # annot_based_clst_of_cellTypeName        0.6500056934396843      150.2137291431427
        # min_dist_clst_of_cellTypeName   0.6366384660773088      18.607512712478638
        # F_1e-2_annot_based_clst_of_cellTypeName 0.6500056934396842
        # F_1e-2_annot_cluster_n3 0.6366384660773088
        # F_1e-4_annot_based_clst_of_cellTypeName 0.6819730299545088
        # F_1e-4_annot_cluster_n3 0.6366384660773088
        # T_1e-4_annot_based_clst_of_cellTypeName 0.5910192545144503
        # T_1e-4_annot_cluster_n3 0.6366384660773088
        # leiden  0.6079559642458384
        # leiden_default  0.6088845879632704
        # louvain_default 0.5937716249547096

# methods_to_keep = [
#     "min_dist_clst_of_cellTypeName",
#     # "F_1e-2_annot_based_clst_of_cellTypeName",
#     "F_1e-4_annot_based_clst_of_cellTypeName",
#     # "T_1e-4_annot_based_clst_of_cellTypeName",
#     "leiden",
#     "leiden_default",
#     "louvain_default"
# ]
methods_to_keep = [
    'Supervised Bonsai-clustering',
    # 'Annot-guided clustering, only big improvements',
    # 'Annot-guided clustering, prohibit small clusters',
    'Leiden',
    # 'Leiden, tutorial-suggested parameters',
    'Louvain', 
    "Unsupervised Bonsai-clustering"
]


df_sub = df[df["method"].isin(methods_to_keep)].copy()
avg_nmi_sub = df_sub.groupby("dataset", sort=False)["nmi"].mean().sort_values()
ordered_datasets_sub = avg_nmi_sub.index.tolist()

xpos = {ds: i for i, ds in enumerate(ordered_datasets_sub)}
df_sub["x"] = df_sub["dataset"].map(xpos)

# Plot
fig, ax = plt.subplots(figsize=(10,5))
for method, sub in df_sub.groupby("method"):
    ax.scatter(sub["x"], sub["nmi"], label=method, alpha=1, s=15)
    

ax.set_xlabel("Dataset (ordered by average NMI)")
ax.set_ylabel("NMI score")
# ax.set_xticks(rotation=45, ha="right")
# ax.set_xticks(range(len(ordered_datasets_sub)),
#            ordered_datasets_sub,
#            rotation=90, ha="right",
#            fontsize=6
#           )

# ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

# Set tick locations
ax.set_xticks(range(len(ordered_datasets_sub)))

# Set tick labels with rotation and formatting
ax.set_xticklabels(
    ordered_datasets_sub,
    rotation=90,
    ha="right",
    fontsize=6
)

ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()

In [ ]:
from pathlib import Path
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/NMI_comparison_raw.svg')
fig.savefig('figures/NMI_comparison_raw.png', dpi=300)